# Text Feature Engineering Assignment
**Dataset:** Real-world product reviews scraped from Amazon  
**Tasks:** Preprocessing, Vocabulary, OHE, BoW, TF-IDF, Sentiment Classification

## Step 0: Install Dependencies

In [ ]:
# Run once if needed
# !pip install requests beautifulsoup4 selenium nltk scikit-learn pandas numpy matplotlib seaborn

## Part A: Data Collection (Web Scraping)

We scrape Amazon product reviews using `requests` + `BeautifulSoup`.  
If scraping is blocked (bot detection), a fallback synthetic dataset is provided automatically.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import os

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

def scrape_amazon_reviews(asin: str, max_pages: int = 10) -> list[dict]:
    """
    Scrape reviews for a given Amazon ASIN.
    Returns a list of dicts with keys: review_text, rating, title.
    """
    reviews = []
    base_url = f"https://www.amazon.in/product-reviews/{asin}/ref=cm_cr_arp_d_paging_btm_next_2"

    for page in range(1, max_pages + 1):
        url = f"{base_url}?pageNumber={page}"
        try:
            resp = requests.get(url, headers=HEADERS, timeout=10)
            if resp.status_code != 200:
                print(f"  Page {page}: HTTP {resp.status_code} - stopping.")
                break
            soup = BeautifulSoup(resp.text, "html.parser")
            blocks = soup.select("div[data-hook='review']")
            if not blocks:
                print(f"  Page {page}: No review blocks found - possible bot block.")
                break
            for block in blocks:
                text_tag = block.select_one("span[data-hook='review-body'] span")
                rating_tag = block.select_one("i[data-hook='review-star-rating'] span")
                title_tag = block.select_one("a[data-hook='review-title'] span")
                if text_tag:
                    reviews.append({
                        "review_text": text_tag.get_text(strip=True),
                        "rating": rating_tag.get_text(strip=True) if rating_tag else "",
                        "title": title_tag.get_text(strip=True) if title_tag else "",
                    })
            print(f"  Page {page}: collected {len(blocks)} reviews (total so far: {len(reviews)})")
            time.sleep(random.uniform(1.5, 3.0))  # polite delay
        except Exception as e:
            print(f"  Page {page}: Error - {e}")
            break
    return reviews


# ---- Fallback: synthetic dataset (used when scraping is blocked) ----
SYNTHETIC_REVIEWS = [
    ("This product is absolutely amazing. Works perfectly and exceeded all my expectations.", 5),
    ("Terrible quality. Broke within a week. Total waste of money.", 1),
    ("Good value for the price. Delivery was fast and packaging was neat.", 4),
    ("Average product. Nothing special but gets the job done.", 3),
    ("Outstanding performance! Best purchase I have made this year.", 5),
    ("Very disappointed. The item does not match the description at all.", 1),
    ("Decent build quality. Would recommend to friends and family.", 4),
    ("Poor customer service and the product stopped working after two days.", 2),
    ("Excellent material and sturdy design. Highly satisfied with this purchase.", 5),
    ("Not worth the price. Found a much better alternative elsewhere.", 2),
    ("Works as advertised. Shipping was on time and product was well packed.", 4),
    ("Very flimsy and cheap looking. Expected better quality at this price.", 2),
    ("Superb product with great features. Totally recommend buying this.", 5),
    ("Instructions were unclear and setup was complicated. Not user friendly.", 2),
    ("Great product for everyday use. Comfortable and lightweight design.", 4),
    ("Received a damaged item. Return process was painful and slow.", 1),
    ("Solid performance and good battery life. Very happy with this.", 5),
    ("Overpriced for what you get. Quality does not justify the cost.", 2),
    ("Nice design and easy to use. Would buy again from this seller.", 4),
    ("Completely useless product. Does not work as described at all.", 1),
    ("Loved the product from day one. Premium feel and great functionality.", 5),
    ("Mediocre quality. Looks good in photos but disappoints in real life.", 3),
    ("Best budget option available. Punches well above its price range.", 4),
    ("Item arrived late and was poorly packaged. Product itself is okay.", 3),
    ("Fantastic value. Does everything I need without any issues whatsoever.", 5),
    ("Too many defects for the price paid. Quality control seems nonexistent.", 1),
    ("Reliable and durable. Has been working great for several months now.", 4),
    ("Feels very cheap and plasticky. Not what I expected from this brand.", 2),
    ("Perfect for my needs. Easy setup and intuitive interface throughout.", 5),
    ("Stopped working after first use. Absolute garbage quality product.", 1),
    ("Good product overall. Minor issues but nothing that ruins the experience.", 4),
    ("Would not recommend. Better products exist at the same price point.", 2),
    ("Impressive performance. Handles demanding tasks with ease and efficiency.", 5),
    ("Battery drains too fast and the charging is unreliable and inconsistent.", 2),
    ("Exactly as described. Happy with the purchase and quick delivery too.", 4),
    ("The product looks nothing like the images. Very misleading listing here.", 1),
    ("Great for the price. Would definitely recommend to anyone on a budget.", 4),
    ("Low quality materials used. Scratches easily and feels very fragile.", 2),
    ("Brilliant product with outstanding durability. Five stars without hesitation.", 5),
    ("Very average and overpriced. Not happy with this purchase decision.", 2),
    ("Quick delivery and product in perfect condition. Very satisfied customer.", 5),
    ("Design is attractive but performance is underwhelming for the cost.", 3),
    ("Works flawlessly. No issues after months of daily use. Highly recommend.", 5),
    ("Faulty product received. Customer support was unhelpful and dismissive.", 1),
    ("Good quality at an affordable price. Shipping was also very prompt.", 4),
    ("This is not worth the money. Very poor quality and bad durability.", 1),
    ("Excellent choice for beginners. Simple to operate and very reliable.", 5),
    ("Product started malfunctioning after a few days of light use only.", 2),
    ("Solid product with no complaints so far. Great value for money overall.", 4),
    ("Wrong item was delivered. Packaging was also damaged during shipping.", 1),
    ("Top notch quality. Well crafted and built to last many years ahead.", 5),
    ("Barely functional. Not even close to meeting basic quality expectations.", 1),
    ("Comfortable and practical. Does exactly what it promises on the label.", 4),
    ("Too many hidden issues. Regret this purchase and would not buy again.", 2),
    ("Excellent product. Fast delivery. Packaged securely. Very happy indeed.", 5),
    ("Stopped working randomly. Very unreliable and inconsistent performance.", 2),
    ("Great product for the price. Works perfectly and looks very nice too.", 5),
    ("Lightweight and portable. Fits easily into my bag without any hassle.", 4),
    ("Defective unit received. No quality check seems to have been done here.", 1),
    ("Very durable and well made. No issues after months of regular usage.", 5),
    ("Loud noise during operation. Expected a quieter and smoother performance.", 2),
    ("Good for casual use. Does not disappoint within its intended use case.", 4),
    ("Material feels cheap. Not suitable for long term or heavy duty use.", 2),
    ("Exceptional product at a fair price. Delivery was ahead of schedule too.", 5),
    ("Multiple missing parts in the box. Absolutely unacceptable from the seller.", 1),
    ("Easy to install and works great right out of the box without issues.", 4),
    ("Overhyped product. Does not live up to the marketing claims at all.", 2),
    ("Amazing purchase. Exactly as expected. Will definitely order again soon.", 5),
    ("Build quality is disappointing. Feels like it will break very soon.", 2),
    ("Good performance and reliable operation. Satisfied with this purchase.", 4),
    ("Very poor product. Falls apart quickly and the quality is substandard.", 1),
    ("Perfect fit for my requirements. Excellent quality and great price too.", 5),
    ("Packaging was damaged on arrival. Product also has visible scratches.", 2),
    ("Smooth and efficient. Meets all my expectations for daily regular usage.", 4),
    ("Completely fake product. Nothing like what was shown in the listing.", 1),
    ("Worth every rupee. High quality product with excellent after sale support.", 5),
    ("Terrible experience overall. Would not recommend this to anyone at all.", 1),
    ("Well designed and functional. A good buy for most average users here.", 4),
    ("Product is fine but delivery took much longer than the estimated date.", 3),
    ("Incredible value for money. Absolutely love every aspect of this product.", 5),
    ("Very thin and fragile. Does not feel like it will survive regular use.", 2),
    ("Works well and is easy to clean and maintain over extended period.", 4),
    ("Completely stopped working after just one week. Terrible product quality.", 1),
    ("Premium look and feel. Performs exceptionally well across all scenarios.", 5),
    ("Not comfortable at all. Causes pain after short periods of use only.", 2),
    ("Reliable and affordable. Does the job perfectly without any complaints.", 4),
    ("Received a used item as new. Very upset with this unethical seller.", 1),
    ("Really love this product. Works consistently and looks great as well.", 5),
    ("Very average product. There are better alternatives in the same range.", 3),
    ("Great investment. Durable and high performance for the price paid here.", 5),
    ("Disappointed with the quality. Expected something much better for this cost.", 2),
    ("Smooth operation and good build. Happy with the overall purchase decision.", 4),
    ("Item was not as described and return policy is too strict and unfair.", 1),
    ("Highly recommend this product. Great quality and amazing customer service.", 5),
    ("Feels very low quality and does not seem like it will last very long.", 2),
    ("Arrived on time and in great condition. Product works perfectly so far.", 4),
    ("Absolute waste of money. The worst product I have bought this entire year.", 1),
    ("Great product with premium quality. Totally satisfied with this purchase.", 5),
    ("Does not work properly. Very frustrating to use and set up correctly.", 1),
    ("Nice product at a reasonable price. Would consider buying again for sure.", 4),
    ("Poor quality product. Very disappointed with this purchase decision overall.", 1),
]

def build_synthetic_dataset() -> pd.DataFrame:
    rows = []
    for text, rating in SYNTHETIC_REVIEWS:
        sentiment = "positive" if rating >= 4 else ("negative" if rating <= 2 else "neutral")
        rows.append({"review_text": text, "rating": rating, "sentiment": sentiment})
    return pd.DataFrame(rows)


# ---- Main collection logic ----
DATASET_PATH = "amazon_reviews.csv"

if os.path.exists(DATASET_PATH):
    print("Loading existing dataset...")
    df = pd.read_csv(DATASET_PATH)
else:
    # Try scraping; ASIN B09G9HD6PD = boAt Rockerz 255 Pro+ (popular, many reviews)
    print("Attempting to scrape Amazon reviews...")
    ASIN = "B09G9HD6PD"
    scraped = scrape_amazon_reviews(ASIN, max_pages=10)

    if len(scraped) >= 100:
        df = pd.DataFrame(scraped)
        # derive numeric rating
        df["rating"] = df["rating"].str.extract(r"(\d)").astype(float)
        df["sentiment"] = df["rating"].apply(
            lambda r: "positive" if r >= 4 else ("negative" if r <= 2 else "neutral")
        )
        print(f"Scraping successful: {len(df)} reviews collected.")
    else:
        print(f"Only {len(scraped)} reviews scraped (need 100). Using synthetic fallback dataset.")
        df = build_synthetic_dataset()

    df.to_csv(DATASET_PATH, index=False)
    print(f"Dataset saved to {DATASET_PATH}")

print(f"\nDataset shape: {df.shape}")
print(df.head(3))

## Task 1: Text Preprocessing

In [ ]:
import re
import string
import nltk

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

STOP_WORDS = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess(text: str, remove_stopwords: bool = True, lemmatize: bool = True) -> str:
    """Full preprocessing pipeline: lowercase -> punctuation removal -> tokenize -> stopwords -> lemmatize."""
    # 1. Lowercase
    text = text.lower()
    # 2. Remove punctuation and special characters
    text = re.sub(r"[^a-z\s]", "", text)
    # 3. Tokenize
    tokens = word_tokenize(text)
    # 4. Remove stopwords (optional)
    if remove_stopwords:
        tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 1]
    # 5. Lemmatize (optional)
    if lemmatize:
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)

df["clean_text"] = df["review_text"].astype(str).apply(preprocess)

print("Sample preprocessing results:")
for i in range(3):
    print(f"\nOriginal : {df['review_text'].iloc[i]}")
    print(f"Processed: {df['clean_text'].iloc[i]}")

## Task 2: Vocabulary Creation

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

# Build vocabulary manually
all_tokens = [token for doc in df["clean_text"] for token in doc.split()]
vocab_counter = Counter(all_tokens)
vocabulary = sorted(vocab_counter.keys())

print(f"Vocabulary size: {len(vocabulary)}")
print(f"Total tokens   : {len(all_tokens)}")

# Top 20 frequent words
top_words = vocab_counter.most_common(20)
print("\nTop 20 most frequent words:")
for word, count in top_words:
    print(f"  {word:20s} {count}")

# Bar chart
words, counts = zip(*top_words)
plt.figure(figsize=(12, 5))
plt.bar(words, counts, color="steelblue")
plt.title("Top 20 Most Frequent Words")
plt.xlabel("Word")
plt.ylabel("Frequency")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("top_words.png", dpi=150)
plt.show()
print("Chart saved as top_words.png")

## Task 3: Feature Engineering

### 3a. One Hot Encoding (document-level)

In [ ]:
import numpy as np

# Use top-N words for OHE to keep matrix manageable
TOP_N = 200
top_vocab = [w for w, _ in vocab_counter.most_common(TOP_N)]
word_to_idx = {w: i for i, w in enumerate(top_vocab)}

def one_hot_encode(doc: str, vocab_index: dict) -> np.ndarray:
    """Binary vector: 1 if word appears in document, 0 otherwise."""
    vec = np.zeros(len(vocab_index), dtype=np.int8)
    for token in set(doc.split()):  # set() -> presence only, not count
        if token in vocab_index:
            vec[vocab_index[token]] = 1
    return vec

ohe_matrix = np.vstack([one_hot_encode(doc, word_to_idx) for doc in df["clean_text"]])

print(f"OHE matrix shape : {ohe_matrix.shape}")
print(f"OHE sparsity     : {(ohe_matrix == 0).sum() / ohe_matrix.size * 100:.2f}%")
print(f"Sample row (first 20 values): {ohe_matrix[0, :20]}")

### 3b. Bag of Words (CountVectorizer)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=500, min_df=2)
bow_matrix = cv.fit_transform(df["clean_text"])

print(f"BoW matrix shape : {bow_matrix.shape}")
print(f"BoW sparsity     : {(1 - bow_matrix.nnz / (bow_matrix.shape[0] * bow_matrix.shape[1])) * 100:.2f}%")

# Top words by total count
bow_word_counts = np.asarray(bow_matrix.sum(axis=0)).flatten()
bow_top_idx = bow_word_counts.argsort()[::-1][:10]
bow_feature_names = cv.get_feature_names_out()
print("\nTop 10 BoW features:")
for idx in bow_top_idx:
    print(f"  {bow_feature_names[idx]:20s} count={int(bow_word_counts[idx])}")

### 3c. TF-IDF (TfidfVectorizer)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=500, min_df=2)
tfidf_matrix = tfidf.fit_transform(df["clean_text"])

print(f"TF-IDF matrix shape : {tfidf_matrix.shape}")
print(f"TF-IDF sparsity     : {(1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])) * 100:.2f}%")

# Words with highest average TF-IDF score
tfidf_mean = np.asarray(tfidf_matrix.mean(axis=0)).flatten()
tfidf_top_idx = tfidf_mean.argsort()[::-1][:10]
tfidf_feature_names = tfidf.get_feature_names_out()
print("\nTop 10 TF-IDF features (by mean score):")
for idx in tfidf_top_idx:
    print(f"  {tfidf_feature_names[idx]:20s} mean_tfidf={tfidf_mean[idx]:.4f}")

## Task 4: Comparison Analysis

In [ ]:
comparison = pd.DataFrame({
    "Aspect": [
        "Vector type",
        "Values",
        "Captures word frequency",
        "Handles rare/common words",
        "Semantic meaning",
        "Typical use case",
        "Sparsity",
    ],
    "One Hot Encoding": [
        "Binary presence vector",
        "0 or 1",
        "No",
        "No",
        "No",
        "Simple classification / small vocab",
        "High",
    ],
    "Bag of Words": [
        "Count vector",
        "Non-negative integers",
        "Yes",
        "No",
        "No",
        "Text classification / topic modelling",
        "High",
    ],
    "TF-IDF": [
        "Weighted float vector",
        "0.0 to 1.0",
        "Yes (normalised)",
        "Yes - penalises common words",
        "No",
        "Information retrieval / ranking",
        "High (but more informative)",
    ],
})

print("Feature Engineering Comparison Table")
print(comparison.to_string(index=False))

In [ ]:
# Why common words get lower TF-IDF weight
print("""
TF-IDF Weight Explanation
==========================
TF-IDF = TF(t, d) * IDF(t)

  TF(t, d)  = frequency of term t in document d / total terms in d
  IDF(t)    = log( N / df(t) ) + 1
              where N = total documents, df(t) = documents containing t

Common words like 'product', 'good', 'buy' appear in almost every review.
Their df(t) is close to N, so IDF approaches log(1) = 0 -> low weight.

Rare but discriminative words like 'shattered', 'flawless', 'defective'
appear in few documents -> high IDF -> high TF-IDF weight.
These words are more informative for distinguishing documents.
""")

## Task 5: Sparse Matrix Analysis

In [ ]:
def sparsity(matrix) -> float:
    """Calculate percentage of zero elements."""
    if hasattr(matrix, 'nnz'):  # scipy sparse
        total = matrix.shape[0] * matrix.shape[1]
        return (1 - matrix.nnz / total) * 100
    else:  # numpy dense
        return (matrix == 0).sum() / matrix.size * 100

print("Sparse Matrix Analysis")
print("-" * 50)
print(f"OHE    shape={ohe_matrix.shape}  sparsity={sparsity(ohe_matrix):.1f}%")
print(f"BoW    shape={bow_matrix.shape}  sparsity={sparsity(bow_matrix):.1f}%")
print(f"TF-IDF shape={tfidf_matrix.shape}  sparsity={sparsity(tfidf_matrix):.1f}%")

print("""
Why sparse matrices are inefficient for large-scale systems
============================================================
1. Memory: A dense float64 matrix of 1M docs x 100K vocab = ~800 GB.
   Even at 98% sparsity, naive storage is impractical.

2. Computation: Matrix multiplications iterate over all cells including zeros,
   wasting CPU/GPU cycles on non-informative values.

3. Solutions in industry:
   - Use scipy.sparse (CSR/CSC format) to store only non-zero values.
   - Switch to dense embeddings (Word2Vec, BERT) which are compact and semantic.
   - Dimensionality reduction: SVD / LSA compresses the sparse matrix.
   - Hashing trick (HashingVectorizer) avoids storing the full vocabulary.
""")

## Task 6: Real-world Questions

In [ ]:
print("""
Q1. Why Bag of Words fails to capture semantic meaning
======================================================
BoW treats every word as an independent dimension with no notion of similarity.

Example:
  Doc A: "The phone battery is excellent."
  Doc B: "The mobile cell power is outstanding."

Semantically these are nearly identical, but their BoW vectors share zero
common features (phone/mobile, battery/power, excellent/outstanding are
different tokens). Cosine similarity = 0, yet meaning is very similar.

BoW also ignores word order, so:
  "The food was good, not bad"  vs  "The food was bad, not good"
produce identical vectors despite opposite sentiment.


Q2. When to use BoW vs TF-IDF in industry
==========================================
Use BoW when:
  - Dataset is small and vocabulary is controlled.
  - Speed matters more than accuracy (fast to compute).
  - Model is Naive Bayes (works well with raw counts).

Use TF-IDF when:
  - Documents vary in length (normalisation is important).
  - Common words should be down-weighted (search engines, document ranking).
  - Building keyword extraction or document similarity systems.
  - Classic text classification pipelines (SVM + TF-IDF is a strong baseline).


Q3. Limitations of TF-IDF in real applications
===============================================
1. No semantic understanding: 'happy' and 'joyful' are unrelated in TF-IDF space.
2. Context-blind: word meaning changes with context (bank: financial vs river).
3. Vocabulary mismatch: unseen words at inference time are silently ignored.
4. High dimensionality: large vocabulary -> very high dimensional sparse vectors.
5. Order agnostic: loses sentence structure and grammar completely.
6. Domain sensitivity: IDF computed on one corpus may not transfer to another.
""")

## Task 7: Mini Use Case - Sentiment Classification

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)

# Filter to binary (positive / negative) for clear classification
binary_df = df[df["sentiment"].isin(["positive", "negative"])].copy().reset_index(drop=True)
print(f"Binary dataset size: {len(binary_df)}")
print(binary_df["sentiment"].value_counts())

X_text = binary_df["clean_text"]
y = binary_df["sentiment"]

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)

# Build BoW and TF-IDF on training split only
cv_cls = CountVectorizer(max_features=500, min_df=1)
X_train_bow = cv_cls.fit_transform(X_train_text)
X_test_bow  = cv_cls.transform(X_test_text)

tfidf_cls = TfidfVectorizer(max_features=500, min_df=1)
X_train_tfidf = tfidf_cls.fit_transform(X_train_text)
X_test_tfidf  = tfidf_cls.transform(X_test_text)

print("\nTrain/test split sizes:")
print(f"  Train: {X_train_bow.shape[0]}  Test: {X_test_bow.shape[0]}")

In [ ]:
results = []

configs = [
    ("Logistic Regression", "BoW",   LogisticRegression(max_iter=1000, random_state=42), X_train_bow,   X_test_bow),
    ("Logistic Regression", "TF-IDF",LogisticRegression(max_iter=1000, random_state=42), X_train_tfidf, X_test_tfidf),
    ("Naive Bayes",         "BoW",   MultinomialNB(),                                     X_train_bow,   X_test_bow),
    ("Naive Bayes",         "TF-IDF",MultinomialNB(),                                     X_train_tfidf, X_test_tfidf),
]

for model_name, feat_name, model, X_train, X_test in configs:
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results.append({"Model": model_name, "Features": feat_name, "Accuracy": round(acc, 4)})
    print(f"\n{model_name} + {feat_name}")
    print(f"  Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred, target_names=["negative", "positive"]))

results_df = pd.DataFrame(results)
print("\nSummary")
print(results_df.to_string(index=False))

In [ ]:
# Confusion matrix for best model (Logistic Regression + TF-IDF typically wins)
lr_tfidf = LogisticRegression(max_iter=1000, random_state=42)
lr_tfidf.fit(X_train_tfidf, y_train)
y_pred_best = lr_tfidf.predict(X_test_tfidf)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_best, labels=["negative", "positive"])
disp = ConfusionMatrixDisplay(cm, display_labels=["negative", "positive"])
disp.plot(ax=axes[0], colorbar=False)
axes[0].set_title("Confusion Matrix: LR + TF-IDF")

# Accuracy bar chart
labels = [f"{r['Model']}\n{r['Features']}" for _, r in results_df.iterrows()]
accs   = results_df["Accuracy"].tolist()
axes[1].bar(labels, accs, color=["steelblue", "darkorange", "steelblue", "darkorange"])
axes[1].set_ylim(0, 1.05)
axes[1].set_title("Model Accuracy Comparison")
axes[1].set_ylabel("Accuracy")
for i, v in enumerate(accs):
    axes[1].text(i, v + 0.01, f"{v:.2%}", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150)
plt.show()
print("Chart saved as model_comparison.png")

## Summary & Observations

In [ ]:
print("""
Observations & Conclusions
===========================

1. Preprocessing significantly reduces vocabulary size and noise.
   Lemmatization further collapses inflected forms (running -> run).

2. OHE captures only word presence. It loses frequency information and
   cannot distinguish a document that uses 'excellent' once vs ten times.

3. BoW improves on OHE by recording raw counts, but treats all words equally.
   Common filler words like 'product' inflate feature vectors without adding
   discriminative power.

4. TF-IDF addresses the common-word problem by scaling down weights for
   high-frequency words across documents. This typically yields better
   classification performance than raw BoW.

5. Logistic Regression generally outperforms Naive Bayes on TF-IDF features
   because LR learns discriminative boundaries, whereas NB assumes feature
   independence (violated in natural language).

6. All three representations produce very sparse matrices (>90% zeros).
   For production systems with millions of documents, sparse matrix formats
   (CSR) or dense embeddings (Word2Vec, BERT) are necessary.

7. None of these representations captures semantic similarity.
   'terrible' and 'awful' remain unrelated vectors. Modern NLP uses
   contextualised embeddings (BERT, RoBERTa) to overcome this.
""")